# EDA Dataset Iris — Panduan Lengkap

**Sub-CPMK:** **P4** — EDA univariat/bivariat; visualisasi dengan Matplotlib & Seaborn.

Notebook ini adalah **tutorial mandiri** (melengkapi `minggu_02.ipynb` dan `minggu_05.ipynb`): eksplorasi tabel, distribusi, pemisahan tiga spesies, dan hipotesis untuk klasifikasi multi-kelas.

**Konteks data:** Diperkenalkan oleh **R.A. Fisher (1936)** untuk demonstrasi klasifikasi. Dataset berisi **150** sampel bunga iris, **50** per spesies, dengan **4** pengukuran morfologi (cm).

**Target:** `species` — tiga kelas: `setosa`, `versicolor`, `virginica`.

**Sumber data:** `sns.load_dataset("iris")` (Seaborn). Versi serupa tersedia di `sklearn.datasets.load_iris` (dipakai `minggu_06`–`minggu_12`).

**Pertanyaan analitik:**
- Apakah keempat fitur berdistribusi simetris?
- Fitur mana yang paling memisahkan **setosa** dari spesies lain?
- Seberapa sulit membedakan **versicolor** vs **virginica**?
- Apakah ada korelasi kuat / redundansi antar fitur?

**Referensi:**
- [Towards Data Science — EDA Walkthrough Iris](https://towardsdatascience.com/an-eda-walkthrough-the-iris-dataset-3f79246266c1/)
- [GeeksforGeeks — EDA on Iris Dataset](https://www.geeksforgeeks.org/data-analysis/exploratory-data-analysis-on-iris-dataset/)
- [ColabCodes — Exploring the Iris Dataset](https://www.colabcodes.com/post/exploring-the-iris-dataset-with-python)
- [Finxter — Easy EDA with Visualization](https://blog.finxter.com/easy-exploratory-data-analysis-eda-in-python-with-visualization/)
- Modul PDF: `modul-05.tex` (Minggu 5 praktikum)

## 0. Kerangka CRISP-DM dan kamus fitur

Fase **Data Understanding** (CRISP-DM): pahami arti kolom sebelum memplot. EDA **iteratif** — temuan di sini mengarahkan pemilihan fitur dan model (`minggu_06`, `minggu_08`).

| Kolom | Arti |
|-------|------|
| `sepal_length` | Panjang sepal (cm) |
| `sepal_width` | Lebar sepal (cm) |
| `petal_length` | Panjang petal (cm) |
| `petal_width` | Lebar petal (cm) |
| `species` | Spesies: setosa, versicolor, virginica |

**Sepal** = kelopak luar; **petal** = kelopak dalam. Pengukuran morfologi ini menjadi prediktor; `species` adalah label untuk klasifikasi terawasi.

## 1. Persiapan lingkungan

Import pustaka standar praktikum dan atur tema Seaborn.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
sns.set_theme(style="whitegrid", context="notebook")
%matplotlib inline

**Memuat data.** `.copy()` agar transformasi tidak mengubah cache global Seaborn.

In [ ]:
df = sns.load_dataset("iris").copy()
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
print("Bentuk data:", df.shape)
df.head()

**Cuplikan akhir, `info`, dan statistik deskriptif.**

In [ ]:
display(df.tail())
df.info()
display(df.describe())

### 1b. Audit kualitas data

Meskipun Iris terkenal "bersih", kebiasaan audit tetap dilakukan (selaras `minggu_03.ipynb`).

In [ ]:
print("Cuplikan acak 20 bunga:")
display(df.sample(20, random_state=RANDOM_STATE))

print("\nMissing per kolom:")
print(df.isna().sum())

print("\nBaris duplikat penuh:", df.duplicated().sum())

print("\nDistribusi kelas species:")
print(df["species"].value_counts())

**Interpretasi (audit):** Biasanya **tidak ada** missing dan **tidak ada** duplikat; ketiga kelas **seimbang** (masing-masing 50 sampel). Dataset siap divisualisasikan tanpa imputasi.

## 2. EDA tabular (Part I)

Ringkasan numerik per spesies dan korelasi antar fitur.

In [ ]:
print("Tipe data:")
print(df.dtypes)

print("\nRata-rata fitur per species (cm):")
means = df.groupby("species", observed=True)[feature_cols].mean().round(2)
display(means)

**Interpretasi (tabel, angka contoh):** `setosa` punya **petal_width** rata-rata ~0,2 cm vs **virginica** ~2,0 cm; **petal_length** setosa ~1,5 cm vs virginica ~5,5 cm. **Sepal** antar versicolor dan virginica lebih mirip — cocok dengan hipotesis bahwa petal lebih diskriminatif.

In [ ]:
corr = df[feature_cols].corr()
display(corr.round(3))

print("Korelasi terkuat antar pasangan fitur:")
pairs = []
for i, a in enumerate(feature_cols):
    for b in feature_cols[i + 1 :]:
        pairs.append((a, b, corr.loc[a, b]))
pairs = sorted(pairs, key=lambda x: abs(x[2]), reverse=True)
for a, b, r in pairs:
    print(f"  {a} vs {b}: {r:.3f}")

**Interpretasi (korelasi):** `petal_length` dan `petal_width` biasanya berkorelasi tinggi (>0,9); fitur petal redundan sebagian — saat modeling bisa uji subset petal saja.

## 3. EDA univariat (Part II — distribusi)

Memahami bentuk distribusi setiap fitur sebelum membandingkan antar spesies.

**Distribusi kelas target.**

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="species", order=["setosa", "versicolor", "virginica"])
plt.title("Jumlah sampel per species")
plt.xlabel("species")
plt.ylabel("Jumlah")
plt.tight_layout()
plt.show()

**Interpretasi:** Kelas **seimbang** — tidak perlu teknik sampling khusus untuk imbalance (berbeda dari Titanic).

**Histogram + KDE keempat fitur.**

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, col in zip(axes.flat, feature_cols):
    sns.histplot(df[col], kde=True, ax=ax, color="steelblue")
    ax.set_title(f"Distribusi {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("Frekuensi")
plt.suptitle("Univariat: keempat fitur (agregat semua species)", y=1.02)
plt.tight_layout()
plt.show()

**Interpretasi:** Agregat semua kelas menghasilkan distribusi **bimodal/multimodal** pada petal (campuran tiga spesies); untuk interpretasi yang benar, bandingkan per `species` di bagian bivariat.

**Boxplot global (cek outlier).** Tutorial TDS menekankan Iris umumnya tanpa outlier ekstrem.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3.5))
for ax, col in zip(axes, feature_cols):
    sns.boxplot(y=df[col], ax=ax, color="lightgray")
    ax.set_title(f"Boxplot {col}")
    ax.set_ylabel(col)
plt.suptitle("Outlier check (semua species digabung)", y=1.05)
plt.tight_layout()
plt.show()

**Interpretasi:** Titik luar IQR sedikit ada pada beberapa fitur tetapi tidak ekstrem; tidak wajib winsorize untuk EDA awal.

## 4. EDA bivariat — fitur vs species

Setiap plot: judul, label sumbu, dan interpretasi singkat (asesmen Minggu 5).

**Boxplot dan violin per fitur.** Membandingkan median dan sebaran antar spesies.

In [ ]:
species_order = ["setosa", "versicolor", "virginica"]
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for i, col in enumerate(feature_cols):
    sns.boxplot(data=df, x="species", y=col, order=species_order, ax=axes[0, i])
    axes[0, i].set_title(f"Boxplot {col}")
    axes[0, i].set_xlabel("species")
    sns.violinplot(data=df, x="species", y=col, order=species_order, ax=axes[1, i])
    axes[1, i].set_title(f"Violinplot {col}")
    axes[1, i].set_xlabel("species")
plt.suptitle("Sebaran fitur per species", y=1.02)
plt.tight_layout()
plt.show()

**Interpretasi:** **Setosa** terpisah jelas pada **petal_length** dan **petal_width**. **Versicolor** vs **virginica** overlap pada sepal; pada petal masih ada pemisahan meski tidak sempurna.

**Scatter: petal_length vs petal_width** — ruang fitur yang paling memisahkan setosa (TDS walkthrough).

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df, x="petal_length", y="petal_width", hue="species",
    hue_order=species_order, palette="Set1", s=60, alpha=0.85,
)
plt.title("Petal length vs petal width (hue: species)")
plt.xlabel("petal_length (cm)")
plt.ylabel("petal_width (cm)")
plt.tight_layout()
plt.show()

**Interpretasi:** **Setosa** membentuk gugus terpisah (petal kecil). **Versicolor** dan **virginica** berdekatan tetapi masih dapat dibedakan sebagian dengan garis batas kasar pada petal_width ≈ 1,6 cm (aturan kasar, bukan kausal).

**Scatter: sepal_length vs sepal_width** — overlap versicolor/virginica lebih besar.

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df, x="sepal_length", y="sepal_width", hue="species",
    hue_order=species_order, palette="Set1", s=60, alpha=0.85,
)
plt.title("Sepal length vs sepal width (hue: species)")
plt.xlabel("sepal_length (cm)")
plt.ylabel("sepal_width (cm)")
plt.tight_layout()
plt.show()

**Interpretasi:** Ketiga spesies **tumpang tindih** di ruang sepal; fitur sepal saja kurang cukup untuk klasifikasi sempurna.

**Stripplot `petal_length`** — melihat sebaran titik per kelas.

In [ ]:
plt.figure(figsize=(8, 4))
sns.stripplot(data=df, x="species", y="petal_length", order=species_order, jitter=0.25, alpha=0.7)
plt.title("Stripplot petal_length per species")
plt.xlabel("species")
plt.ylabel("petal_length (cm)")
plt.tight_layout()
plt.show()

**Interpretasi:** Pemisahan vertikal antar kelas pada `petal_length` terlihat jelas; setosa seluruhnya di bawah ~2 cm.

**FacetGrid: histogram per species untuk `petal_width`.**

In [ ]:
g = sns.FacetGrid(df, col="species", col_order=species_order, height=3, sharex=True)
g.map_dataframe(sns.histplot, x="petal_width", kde=True, bins=15)
g.set_axis_labels("petal_width (cm)", "Frekuensi")
g.fig.suptitle("Distribusi petal_width per species", y=1.05)
plt.show()

**Interpretasi:** Tiga distribusi `petal_width` hampir tidak tumpang tindih — fitur kuat untuk model.

## 5. EDA multivariat

**Catatan:** Korelasi Pearson pada fitur morfologi; bukan bukti kausalitas biologis.

**Heatmap korelasi.**

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(df[feature_cols].corr(), annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Heatmap korelasi Pearson (4 fitur)")
plt.tight_layout()
plt.show()

**Interpretasi:** Korelasi negatif lemah `sepal_width` vs petal; petal saling berkorelasi kuat — pertimbangkan regularisasi atau seleksi fitur.

**Pairplot** — hubungan pasangan semua fitur, diwarnai species.

In [ ]:
sns.pairplot(df, vars=feature_cols, hue="species", hue_order=species_order, corner=True, plot_kws={"alpha": 0.7})
plt.show()

**Interpretasi:** Panel petal menunjukkan pemisahan kelas terbaik; panel sepal menunjukkan overlap versicolor–virginica — konsisten dengan scatter di atas.

**Pairplot subset petal saja** — fokus fitur kandidat untuk modeling.

In [ ]:
sns.pairplot(
    df, vars=["petal_length", "petal_width"], hue="species",
    hue_order=species_order, corner=True, plot_kws={"alpha": 0.75},
)
plt.show()

**Interpretasi:** Dua fitur petal saja sudah memisahkan setosa hampir sempurna; cukup untuk baseline klasifikasi multi-kelas.

## 6. (Opsional) Bandingkan Seaborn vs scikit-learn

Memastikan konsistensi dengan notebook `minggu_06` ke atas.

In [ ]:
from sklearn.datasets import load_iris

iris_sk = load_iris(as_frame=True)
rename_sk = {
    "sepal length (cm)": "sepal_length",
    "sepal width (cm)": "sepal_width",
    "petal length (cm)": "petal_length",
    "petal width (cm)": "petal_width",
}
df_sk = iris_sk.frame.rename(columns=rename_sk)
df_sk["species"] = iris_sk.target_names[iris_sk.target]
print("Seaborn shape:", df.shape, "| sklearn shape:", df_sk.shape)
print("Kolom Seaborn:", df.columns.tolist())
print("Kolom sklearn (setelah rename):", [c for c in df_sk.columns if c != "target"])
print("\nRata petal_length setosa (Seaborn):", df.loc[df.species == "setosa", "petal_length"].mean().round(3))
print("Rata petal_length setosa (sklearn):", df_sk.loc[df_sk.species == "setosa", "petal_length"].mean().round(3))

**Interpretasi:** Nilai numerik setara; perbedaan hanya penamaan kolom target. Gunakan satu sumber konsisten dalam satu proyek.

## 7. Ringkasan temuan, bias, dan hipotesis pemodelan

### Ringkasan temuan (EDA)

1. **150 sampel, 3 kelas seimbang** (50 per `species`); tidak ada missing pada versi Seaborn.
2. **Setosa** morfologis terpisah, terutama pada **petal_length** dan **petal_width**.
3. **Versicolor** vs **virginica** sulit dibedakan lewat **sepal**; **petal** memberi pemisahan lebih baik.
4. **Korelasi tinggi** antara `petal_length` dan `petal_width` — redundansi parsial.
5. **Outlier** tidak dominan; boxplot global relatif kompak.
6. **Pairplot** mengonfirmasi: ruang 2D petal hampir cukup untuk klasifikasi tiga kelas.

### Bias dan limitasi

- Dataset **kecil, bersih, seimbang** — tidak merepresentasikan data industri (noise, imbalance, drift).
- Pengukuran historis; generalisasi ke populasi bunga lain tidak otomatis valid.
- EDA **iteratif** (CRISP-DM): setelah baseline model, kembali ke plot jika akurasi versicolor/virginica rendah.

### Hipotesis pemodelan

Klasifikasi **3 kelas** dengan **petal_length** dan **petal_width** saja diharapkan mencapai akurasi sangat tinggi (setosa ~100% pada uji); menambahkan **sepal** mungkin marginally membantu atau redundan — uji di `minggu_08.ipynb` dengan membandingkan subset fitur. **K-Means** dengan \(k=3\) pada fitur (tanpa label) di `minggu_10.ipynb` seharusnya mendekati struktur alami data.

### Langkah berikutnya

1. `minggu_06.ipynb` — `train_test_split`, `fit`/`score`
2. `minggu_08.ipynb` — LogReg, KNN, pohon + confusion matrix
3. `minggu_10.ipynb` — K-Means, siluet

Jalankan **Kernel → Restart & Run All** pada notebook ini.